# NIDS hybride auto-supervisé sur CIC-IDS2017 : version exécutable sous Google Colab

E-GraphSAGE attentionnel + pré-entraînement contrastif (GCL) + tête temporelle LSTM + XGBoost (imitateur) + explicabilité (TreeSHAP, GNNExplainer) + inférence en ligne, robustesse et dérive.

**Avant de commencer**
1. Menu *Exécution > Modifier le type d'exécution* : choisir un **GPU** (T4 suffit).
2. Sur Google Drive, créer le dossier `MyDrive/NIDS/` et y déposer :
   - `NIDS_Colab.zip` (l'archive qui contient ce carnet et le dossier `nids/`) ;
   - un sous-dossier `csv/` avec les 8 fichiers CSV de CIC-IDS2017 **avec les colonnes `Flow ID`, `Source IP`, `Destination IP`, `Timestamp`** (version « TrafficLabelling »).
3. Exécuter les cellules **dans l'ordre**, de haut en bas. Chaque étape saute son calcul si ses sorties existent déjà (reprise après déconnexion). Pour tout refaire : `FORCE = True`.

**Stockage** : le calcul se fait sur le disque local de Colab (rapide). Après chaque étape, les nouveaux fichiers de `data/`, `models/` et `reports/` sont copiés sur Drive dans `MyDrive/NIDS/work/` ; au démarrage d'une nouvelle session, ils sont recopiés de Drive vers le disque local.

## 0. Paramètres

In [ ]:
DRIVE_BASE = "/content/drive/MyDrive/NIDS"      # dossier de travail sur Drive
CSV_DIR    = f"{DRIVE_BASE}/csv"                # les 8 CSV de CIC-IDS2017 (copiés sur le disque local au démarrage)
ZIP_PATH   = f"{DRIVE_BASE}/NIDS_Colab.zip"     # archive du projet (si absente, le carnet demande un envoi manuel)

SEEDS  = [0]            # [0, 1, 2, 3, 4] pour les statistiques multi-graines (vote, stabilité, HS1) : 5 fois plus long
DEVICE = "cuda"         # "cuda" (GPU Colab) ou "cpu"
SMOKE  = False          # True : essai très court des entraînements (vérifie la chaîne). Après un essai, repasser SMOKE=False ET FORCE=True
FORCE  = False          # True : ignore les sorties existantes et recalcule tout
REFRESH_CODE = False    # True : réextrait le code (src/, scripts/) de l'archive sans toucher aux données, modèles et rapports

LABEL_FRACTION = 0.1    # fraction d'étiquettes de la configuration principale (10 %)
MIN_LABELS = 0          # plancher d'étiquettes PAR CLASSE (0 = protocole d'origine). Ex. 50 : chaque classe a au moins 50 flux étiquetés
                        # (ou tous ses flux si elle en a moins). Les classes rares (Infiltration, SQL Injection, Heartbleed) ne sont plus réduites à 1-3 exemples.
EXPERIMENT = ""         # suffixe du dossier de résultats. OBLIGATOIRE si MIN_LABELS > 0 (ex. "_m50") : les résultats de référence ne sont pas écrasés.
                        # Au premier lancement, graphes, données normalisées et encodeurs pré-entraînés (sans étiquettes) sont repris de la référence.
assert MIN_LABELS == 0 or EXPERIMENT, 'MIN_LABELS > 0 demande un suffixe, par exemple EXPERIMENT = "_m50"'
MLAB = f" --min-labels-per-class {MIN_LABELS}" if MIN_LABELS else ""
HEAD_SEED = SEEDS[0]    # graine de la tête utilisée pour l'explicabilité, le temps réel et la robustesse

## 1. Installation, montage de Drive, préparation du projet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys, shutil, zipfile, subprocess, glob, time, json, re, importlib.util
from pathlib import Path

# Colab fournit déjà torch, numpy, pandas, scikit-learn, matplotlib, psutil : on n'installe que ce qui manque
# (sans mise à jour des paquets existants, pour ne pas casser les binaires déjà chargés).
_missing = [p for p, m in [("xgboost", "xgboost"), ("pyarrow", "pyarrow"), ("networkx", "networkx"),
                           ("psutil", "psutil")] if importlib.util.find_spec(m) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

ROOT = f"/content/NIDS_Colab{EXPERIMENT}"
PROJECT = f"{ROOT}/nids"
WORK_DRIVE = f"{DRIVE_BASE}/work{EXPERIMENT}"   # copie durable des sorties sur Drive
BASE_WORK_DRIVE = f"{DRIVE_BASE}/work"           # résultats de référence (reprise des entrées communes d'une expérience)
SYNC_DIRS = ["data", "models", "reports"]
CSV_LOCAL = "/content/nids_csv"

def _extract_project():
    zp = ZIP_PATH
    if not os.path.exists(zp):
        from google.colab import files
        print("Archive absente de Drive : envoyez NIDS_Colab.zip.")
        up = files.upload()
        zp = f"/content/{list(up)[0]}"
    tmp = "/content/_nids_unzip"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(tmp)
    # l'archive peut contenir NIDS_Colab/nids/ ou directement nids/ : on cherche le dossier qui contient src/
    cands = [dp for dp, dns, _ in os.walk(tmp) if "src" in dns and os.path.isdir(os.path.join(dp, "src", "ingestion"))]
    assert cands, f"aucun dossier src/ trouvé dans l'archive {zp}"
    src = min(cands, key=len)
    os.makedirs(PROJECT, exist_ok=True)
    for item in os.listdir(src):            # code et fichiers du projet ; données/modèles/rapports locaux jamais écrasés
        if item in SYNC_DIRS and os.path.isdir(f"{PROJECT}/{item}"):
            continue
        dst = f"{PROJECT}/{item}"
        if os.path.isdir(f"{src}/{item}"):
            shutil.copytree(f"{src}/{item}", dst, dirs_exist_ok=True)
        else:
            shutil.copy2(f"{src}/{item}", dst)
    shutil.rmtree(tmp, ignore_errors=True)
    # src/ doit être un paquet Python
    for dp, dns, fns in os.walk(f"{PROJECT}/src"):
        dns[:] = [d for d in dns if d != "__pycache__"]
        if "__init__.py" not in fns:
            open(os.path.join(dp, "__init__.py"), "a").close()

if REFRESH_CODE or not os.path.isdir(f"{PROJECT}/src"):
    _extract_project()
assert os.path.isdir(f"{PROJECT}/src"), f"projet introuvable : {PROJECT}"

# Nettoyage : une session précédente (ancienne version du carnet) a pu laisser des raccourcis (liens) dans le projet.
# On les supprime (le lien seul, jamais son contenu sur Drive) pour ne garder que de vrais dossiers.
def _remove_links(root):
    n = 0
    for dp, dns, fns in os.walk(root):
        for name in list(dns) + list(fns):
            q = os.path.join(dp, name)
            if os.path.islink(q):
                os.unlink(q); n += 1
                if name in dns: dns.remove(name)
    return n
print("Liens supprimés :", _remove_links(PROJECT))

# Dossiers de travail : de vrais dossiers locaux ; Drive sert de sauvegarde.
for d in SYNC_DIRS + ["models/checkpoints", "reports/figures", "reports/metrics", "reports/logs", "data/interim", "data/embeddings"]:
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)

def _copy_tree(src_root, dst_root, label=""):
    """Copie incrémentale (taille + date) de src_root vers dst_root ; renvoie le nombre de fichiers copiés."""
    n = 0
    for dp, _, fns in os.walk(src_root):
        for fn in fns:
            s = os.path.join(dp, fn)
            d = os.path.join(dst_root, os.path.relpath(s, src_root))
            try:
                ss = os.stat(s)
                if os.path.exists(d):
                    ds = os.stat(d)
                    if ds.st_size == ss.st_size and abs(ds.st_mtime - ss.st_mtime) < 3:
                        continue
                os.makedirs(os.path.dirname(d), exist_ok=True)
                for attempt in range(3):
                    try:
                        shutil.copy2(s, d); break
                    except OSError:
                        if attempt == 2: raise
                        time.sleep(2)
                n += 1
            except FileNotFoundError:
                pass
    return n

def pull_from_drive():
    """Drive -> disque local (reprise après une nouvelle session)."""
    n = 0
    for d in SYNC_DIRS:
        if os.path.isdir(f"{WORK_DRIVE}/{d}"):
            n += _copy_tree(f"{WORK_DRIVE}/{d}", f"{PROJECT}/{d}")
    return n

def push_to_drive():
    """Disque local -> Drive (sauvegarde après chaque étape)."""
    n = 0
    for d in SYNC_DIRS:
        os.makedirs(f"{WORK_DRIVE}/{d}", exist_ok=True)
        n += _copy_tree(f"{PROJECT}/{d}", f"{WORK_DRIVE}/{d}")
    return n

def seed_from_base():
    """Nouvelle expérience : reprend de la référence ce qui ne dépend pas du plancher d'étiquettes (aucune étiquette n'y intervient)."""
    if not EXPERIMENT:
        return 0
    n = 0
    for sub in ["data/graphs", "data/raw/csvnormalisees", "models/checkpoints/gcl_att.pt", "models/checkpoints/egnn_att.pt",
                "reports/normalisation.md", "reports/graph_store.md", "reports/split_dryrun.md", "reports/episodes.md",
                "reports/audit_raw.md", "reports/clean_dryrun.md"]:
        s_, d_ = f"{BASE_WORK_DRIVE}/{sub}", f"{PROJECT}/{sub}"
        if os.path.isdir(s_):
            n += _copy_tree(s_, d_)
        elif os.path.isfile(s_) and not os.path.exists(d_):
            os.makedirs(os.path.dirname(d_), exist_ok=True); shutil.copy2(s_, d_); n += 1
    return n

print("Restauration depuis Drive :", pull_from_drive(), "fichier(s) copié(s)")
if EXPERIMENT:
    print("Entrées reprises de la référence :", seed_from_base(), "fichier(s)")

# CSV : copie locale unique (lecture plus rapide et plus fiable que Drive)
assert os.path.isdir(CSV_DIR) and glob.glob(f"{CSV_DIR}/*.csv"), f"aucun CSV dans {CSV_DIR}"
os.makedirs(CSV_LOCAL, exist_ok=True)
_copy_tree(CSV_DIR, CSV_LOCAL)
CSV_DIR_DRIVE, CSV_DIR = CSV_DIR, CSV_LOCAL     # la suite du carnet lit les CSV locaux
print("CSV trouvés :", len(glob.glob(f"{CSV_DIR}/*.csv")))
print("Projet :", PROJECT)

In [ ]:
import torch
print("torch", torch.__version__, "| GPU disponible :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")
if DEVICE == "cuda" and not torch.cuda.is_available():
    print("ATTENTION : pas de GPU, passage automatique en CPU (très lent).")
    DEVICE = "cpu"

def _done(skip_if):
    if not skip_if or FORCE:
        return False
    pats = [skip_if] if isinstance(skip_if, str) else skip_if
    return all(glob.glob(f"{PROJECT}/{p}") for p in pats)

def run(cmd, skip_if=None, name=None):
    """Lance une commande depuis la racine du projet, affiche la sortie en direct, journalise, s'arrête en cas d'erreur."""
    if _done(skip_if):
        print(f"[déjà fait] {cmd.split()[1] if len(cmd.split()) > 1 else cmd}  (sorties présentes)")
        return
    print(f"$ {cmd}")
    env = dict(os.environ, PYTHONPATH=PROJECT, PYTHONUNBUFFERED="1")
    t0 = time.time()
    log = f"{PROJECT}/reports/logs/{name or re.sub(r'[^A-Za-z0-9_.]+', '_', cmd)[:80]}.log"
    with open(log, "w", encoding="utf-8") as lf:
        p = subprocess.Popen(cmd, shell=True, cwd=PROJECT, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); lf.write(line)
        p.wait()
    print(f"-> terminé en {time.time() - t0:.0f} s (code {p.returncode})")
    if p.returncode != 0:
        tail = "".join(open(log, encoding="utf-8", errors="replace").readlines()[-25:])
        raise RuntimeError(f"échec : {cmd}\n--- 25 dernières lignes (journal : {log}) ---\n{tail}")
    print(f"   sauvegarde Drive : {push_to_drive()} fichier(s)")

SM = " --smoke" if SMOKE else ""

## 2. Étape 1 : audit, nettoyage et normalisation des flux
Scripts : `scripts/audit/01_audit_raw.py` → `02_clean_dryrun.py` (simulation puis `--apply`) → `03_episodes.py`, `04_split_dryrun.py` (contrôles) → `src/ingestion/normalize.py` → `scripts/audit/05_features_check.py`.

In [ ]:
run(f"python scripts/audit/01_audit_raw.py --raw-dir {CSV_DIR}", skip_if="reports/audit_raw.md")
run(f"python scripts/audit/02_clean_dryrun.py --raw-dir {CSV_DIR}", skip_if="reports/clean_dryrun.md")
# nettoyage réellement appliqué : écrit des Parquet dans data/interim
run(f"python scripts/audit/02_clean_dryrun.py --raw-dir {CSV_DIR} --apply --out-dir data/interim", skip_if="data/interim/*.parquet", name="clean_apply")
run("python scripts/audit/03_episodes.py --in-dir data/interim", skip_if="reports/episodes.md")
run("python scripts/audit/04_split_dryrun.py --in-dir data/interim", skip_if="reports/split_dryrun.md")

In [ ]:
# normalisation (statistiques de l'entraînement seul) et découpage temporel 70 / 10 / 20 %
run("python -m src.ingestion.normalize --in-dir data/interim --out-dir data/raw/csvnormalisees",
    skip_if=["data/raw/csvnormalisees/train.parquet", "data/raw/csvnormalisees/scaler.json"])
run("python scripts/audit/05_features_check.py --dir data/raw/csvnormalisees", skip_if="reports/features_check.md")

## 3. Étape 2 : magasin de graphes temporels (un graphe par fenêtre d'une minute)

In [ ]:
run("python -m src.ingestion.graph_store --in-dir data/raw/csvnormalisees --out-dir data/graphs",
    skip_if=["data/graphs/train.npz", "data/graphs/val.npz", "data/graphs/test.npz", "data/graphs/meta.json"])

## 4. Référence sans graphe : XGBoost sur les 63 caractéristiques (optionnel)

In [ ]:
run(f"python -m src.models.baseline_xgb --device {DEVICE} --label-fraction {LABEL_FRACTION}{MLAB} --tag _f10", skip_if="reports/baseline_xgb_f10.md")
run(f"python -m src.models.baseline_xgb --device {DEVICE} --label-fraction {LABEL_FRACTION}{MLAB} --use-dport --tag _dport", skip_if="reports/baseline_xgb_dport.md")

## 5. Étape 3 : E-GraphSAGE attentionnel et pré-entraînement contrastif
- `train_egnn` : encodeur supervisé de bout en bout (comparaison).
- `pretrain_gcl` : pré-entraînement auto-supervisé (aucune étiquette).
- `finetune` : classification avec 10 % d'étiquettes, encodeur pré-entraîné dégelé (une exécution par graine).

In [ ]:
run(f"python -m src.models.egnn.train_egnn --device {DEVICE} --tag _att{SM}", skip_if="models/checkpoints/egnn_att.pt")
run(f"python -m src.models.gcl.pretrain_gcl --device {DEVICE} --tag _att{SM}", skip_if="models/checkpoints/gcl_att.pt")
for s in SEEDS:
    run(f"python -m src.models.gcl.finetune --device {DEVICE} --encoder models/checkpoints/gcl_att.pt --unfreeze "
        f"--label-fraction {LABEL_FRACTION}{MLAB} --seed {s} --tag _unfrozen_f10_s{s}{SM}", skip_if=f"models/checkpoints/ft_unfrozen_f10_s{s}.pt")

In [ ]:
# OPTIONNEL : fractions d'étiquettes 1, 5, 100 % (graine 0), encodeur gelé et dégelé (étude HS2)
for pct, frac in [(1, 0.01), (5, 0.05), (10, 0.1), (100, 1.0)]:
    for mode, flag in [("frozen", ""), ("unfrozen", " --unfreeze")]:
        run(f"python -m src.models.gcl.finetune --device {DEVICE} --encoder models/checkpoints/gcl_att.pt{flag} "
            f"--label-fraction {frac}{MLAB} --seed 0 --tag _{mode}_f{pct}_s0{SM}", skip_if=f"models/checkpoints/ft_{mode}_f{pct}_s0.pt")

## 6. Étape 4 : tête temporelle LSTM (historique de 5 minutes par hôte)

In [ ]:
for s in SEEDS:
    extra = " --save-embeddings" if s == SEEDS[0] else ""
    run(f"python -m src.models.lstm.train_temporal --device {DEVICE} --checkpoint models/checkpoints/ft_unfrozen_f10_s{s}.pt "
        f"--label-fraction {LABEL_FRACTION}{MLAB} --seed {s} --tag _tl_f10_s{s} --save-preds --save-logits{extra}{SM}",
        skip_if=[f"models/checkpoints/temporal_tl_f10_s{s}.pt", f"data/embeddings/pred_tl_f10_s{s}_test.npy"])

In [ ]:
# Courbes par époque (perte d'entraînement, perte et exactitude de validation et de test) : nouvel entraînement, tag _cv_s0
run(f"python -m src.models.lstm.train_temporal --device {DEVICE} --checkpoint models/checkpoints/ft_unfrozen_f10_s{HEAD_SEED}.pt "
    f"--label-fraction {LABEL_FRACTION}{MLAB} --seed {HEAD_SEED} --tag _cv_s0 --save-logits --curves{SM}", skip_if="reports/curves_cv_s0.csv")

## 7. Étape 5 : XGBoost (fusion avec les représentations) et imitateur de la tête

In [ ]:
HEAD = f"models/checkpoints/temporal_tl_f10_s{HEAD_SEED}.pt"
PRED = f"_tl_f10_s{HEAD_SEED}"
run(f"python -m src.models.xgb_fusion --device {DEVICE} --emb-tag {PRED} --features raw+emb --label-fraction {LABEL_FRACTION}{MLAB} --seed {HEAD_SEED} --tag _f10_s{HEAD_SEED}",
    skip_if=f"reports/xgbfusion_f10_s{HEAD_SEED}.md")
# imitateur (modèle d'explication) puis réglage sur la validation
run(f"python -m src.models.surrogate_xgb --device {DEVICE} --pred-tag {PRED} --tag _s{HEAD_SEED}", skip_if=f"models/checkpoints/surrogate_s{HEAD_SEED}.json")
run(f"python -m src.models.surrogate_tune --device {DEVICE} --pred-tag {PRED} --tag _s{HEAD_SEED}", skip_if=f"models/checkpoints/surrogate_tuned_s{HEAD_SEED}.json")

## 8. Étape 6 : évaluation, stabilité et optimisation
`confusion_analysis` → `figures_eval` (matrice de confusion, ROC, courbes par époque) → statistiques multi-graines (si `SEEDS` en compte au moins deux).

In [ ]:
run(f"python -m src.eval.confusion_analysis --pred-tag {PRED} --split test --tag _s{HEAD_SEED}", skip_if=f"reports/confusion_s{HEAD_SEED}.md")
run(f"python -m src.eval.figures_eval --logit-tag {PRED} --tag _s{HEAD_SEED}", skip_if=f"reports/figures/confusion_s{HEAD_SEED}.png")
run("python -m src.eval.figures_eval --logit-tag _cv_s0 --curves-tag _cv_s0 --tag _cv_s0", skip_if="reports/figures/courbes_cv_s0.png")
run(f"python -m src.eval.tune_bias --logit-tag {PRED} --label-fraction {LABEL_FRACTION}{MLAB} --seed {HEAD_SEED} --max-fpr 0.001", skip_if=f"reports/tune_bias.md")
if len(SEEDS) >= 2:
    sd = " ".join(map(str, SEEDS))
    run(f"python -m src.eval.aggregate_attack --pattern '_tl_f10_s{{}}' --seeds {sd} --min-support 100 --tag _tl_f10", skip_if="reports/attack_seeds_tl_f10.md")
    run(f"python -m src.eval.stability --pattern '_tl_f10_s{{}}' --seeds {sd} --min-seeds {min(3, len(SEEDS))} --tag _tl_f10", skip_if="reports/stability_tl_f10.md")
    run(f"python -m src.eval.vote --pattern '_tl_f10_s{{}}' --seeds {sd} --min-support 100 --tag _tl_f10 --save-tag _vote{len(SEEDS)}", skip_if="reports/vote_tl_f10.md")
    run(f"python -m src.eval.hs1_classes --pattern '_tl_f10_s{{}}' --seeds {','.join(map(str, SEEDS))} --tag _tl_f10", skip_if="reports/hs1_classes_tl_f10.md")
    run(f"python -m src.eval.label_coverage --pattern '_tl_f10_s{{}}' --seeds {sd} --label-fraction {LABEL_FRACTION}{MLAB} --tag _tl_f10", skip_if="reports/label_coverage_tl_f10.md")
else:
    print("Une seule graine : les statistiques multi-graines sont ignorées (SEEDS = [0, 1, 2, 3, 4] pour les produire).")

## 9. Étape 7 : inférence en ligne (rejeu fenêtre par fenêtre), latence, inductivité (EF2)

In [ ]:
run(f"python -m src.realtime.replay --device {DEVICE} --head {HEAD} --pred-file data/embeddings/pred{PRED}_test.npy --tag _s{HEAD_SEED}", skip_if=f"reports/replay_s{HEAD_SEED}.md")
run(f"python -m src.eval.benchmark_latency --device {DEVICE} --surrogate-device {DEVICE} --head {HEAD} --pred-file data/embeddings/pred{PRED}_test.npy "
    f"--surrogate models/checkpoints/surrogate_s{HEAD_SEED}.json", skip_if="reports/latency.md")
run(f"python -m src.eval.ef2_hosts --device {DEVICE} --head {HEAD} --pred-file data/embeddings/pred{PRED}_test.npy --tag _s{HEAD_SEED}", skip_if=f"reports/ef2_hosts_s{HEAD_SEED}.md")
if len(SEEDS) >= 2:
    run(f"python -m src.eval.ef2_novelty --pattern '_tl_f10_s{{}}' --seeds {','.join(map(str, SEEDS))} --tag _tl_f10", skip_if="reports/ef2_novelty_tl_f10.md")

## 10. Étape 8 : explicabilité (TreeSHAP, GNNExplainer), robustesse, dérive
Le réglage de GNNExplainer (`gnn_tune`) se fait sur la **validation** ; il écrit `reports/xai_gnn_tune_s*.md`. Relire ce rapport, puis reporter les coefficients retenus dans `GNN_ARGS` (forme : `--size-coef X --ent-coef Y --budget-coef Z --budget-frac W`) et relancer la cellule de l'explicateur avec `FORCE = True`. Par défaut, l'explicateur utilise les valeurs par défaut du script.

In [ ]:
run(f"python -m src.models.xai.shap_fidelity --device {DEVICE} --head {HEAD} --surrogate models/checkpoints/surrogate_s{HEAD_SEED}.json --pred-tag {PRED} --tag _s{HEAD_SEED}",
    skip_if=f"reports/xai_shap_s{HEAD_SEED}.md")
run(f"python -m src.models.xai.shap_fidelity --device {DEVICE} --head {HEAD} --surrogate models/checkpoints/surrogate_tuned_s{HEAD_SEED}.json --pred-tag {PRED} --tag _s{HEAD_SEED}cls_tuned",
    skip_if=f"reports/xai_shap_s{HEAD_SEED}cls_tuned.md")
run(f"python -m src.models.xai.gnn_tune --device {DEVICE} --head {HEAD} --pred-tag {PRED} --tag _s{HEAD_SEED}", skip_if=f"reports/xai_gnn_tune_s{HEAD_SEED}.md")
GNN_ARGS = ""   # à remplir d'après reports/xai_gnn_tune_s0.md, ex. "--size-coef X --ent-coef Y --budget-coef Z --budget-frac W"
run(f"python -m src.models.xai.gnn_explainer --device {DEVICE} --head {HEAD} --pred-tag {PRED} {GNN_ARGS} --tag _s{HEAD_SEED}_gnn",
    skip_if=f"reports/xai_gnn_s{HEAD_SEED}_gnn.md")
run(f"python -m src.models.xai.xai_figures --shap-tag _s{HEAD_SEED} --gnn-tag _s{HEAD_SEED}_gnn --logit-tag {PRED} --tag _s{HEAD_SEED}", skip_if=f"reports/figures/shap_summary_s{HEAD_SEED}.png")

In [ ]:
# robustesse aux perturbations aléatoires (fantômes, bruit, suppression de flux, perte de fenêtres) puis surveillance de la dérive
run(f"python -m src.realtime.adversarial --device {DEVICE} --head {HEAD} --tag _s{HEAD_SEED}", skip_if=f"reports/adversarial_s{HEAD_SEED}.md")
run(f"python -m src.realtime.drift --device {DEVICE} --head {HEAD} --pred-tag {PRED} --tag _s{HEAD_SEED}", skip_if=f"reports/drift_s{HEAD_SEED}.md")

## 11. Diagnostics HS1 (optionnel)
Tests de volume, de contexte et d'ordre sur la classe DDoS pour une tête donnée (le document les mène sur la graine 5). Indiquer `DIAG_SEED` parmi les graines entraînées.

In [ ]:
DIAG_SEED = HEAD_SEED
dh = f"models/checkpoints/temporal_tl_f10_s{DIAG_SEED}.pt"
run(f"python -m src.realtime.volume_test --device {DEVICE} --head {dh} --class-name DDoS --tag _ddos_s{DIAG_SEED}", skip_if=f"reports/volume_ddos_s{DIAG_SEED}.md")
run(f"python -m src.realtime.context_test --device {DEVICE} --head {dh} --class-name DDoS --tag _ddos_s{DIAG_SEED}", skip_if=f"reports/context_ddos_s{DIAG_SEED}.md")
run(f"python -m src.realtime.order_test --device {DEVICE} --head {dh} --class-name DDoS --tag _s{DIAG_SEED}_{DEVICE}", skip_if=f"reports/order_s{DIAG_SEED}_{DEVICE}.md")

---
# Affichage des métriques du document
Chaque cellule ci-dessous correspond à une section du document de méthodologie. Une étape non exécutée est signalée « non disponible ».

In [ ]:
from IPython.display import Markdown, Image, display
import matplotlib.pyplot as plt

def show_md(name, max_chars=None):
    p = f"{PROJECT}/reports/{name}"
    if not os.path.exists(p):
        print(f"[non disponible] reports/{name}"); return
    t = open(p, encoding="utf-8").read()
    if max_chars and len(t) > max_chars:
        t = t[:max_chars] + "\n\n[... tronqué ; fichier complet : reports/" + name + "]"
    display(Markdown(t))

def show_img(name):
    p = f"{PROJECT}/reports/figures/{name}"
    if not os.path.exists(p):
        print(f"[non disponible] reports/figures/{name}"); return
    display(Image(filename=p))

def parse_curve(name):
    """Lit le tableau « Époque | ... » d'un rapport d'entraînement ; renvoie (en-têtes, lignes numériques)."""
    p = f"{PROJECT}/reports/{name}"
    if not os.path.exists(p):
        return None, []
    rows, head = [], None
    for line in open(p, encoding="utf-8"):
        c = [x.strip() for x in line.strip().strip("|").split("|")]
        if c and c[0] == "Époque":
            head = c
        elif head and c and c[0].isdigit() and len(c) == len(head):
            rows.append([float(x) for x in c])
        elif head and rows and not line.startswith("|"):
            break
    return head, rows

### Section 2 — Données et graphes

In [ ]:
for n in ["normalisation.md", "graph_store.md", "split_dryrun.md"]:
    show_md(n, 6000)

### Section 5 — Résultats de détection : métriques, matrice de confusion, ROC

In [ ]:
show_md(f"confusion_s{HEAD_SEED}.md", 9000)
show_img(f"confusion_s{HEAD_SEED}.png")
show_md(f"roc_s{HEAD_SEED}.md")
show_img(f"roc_s{HEAD_SEED}.png")

### Section 6 — Courbes d'apprentissage (perte d'entraînement, validation, test, exactitude)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
h, r = parse_curve("gcl_att.md")
if r:
    ax[0].plot([x[0] for x in r], [x[1] for x in r], label="entraînement")
    ax[0].plot([x[0] for x in r], [x[2] for x in r], label="validation")
    ax[0].set_title("Perte contrastive (GCL)"); ax[0].set_xlabel("Époque"); ax[0].legend(); ax[0].grid(alpha=.3)
for lab, f in [("E-GraphSAGE supervisé", "egnn_att.md"), ("Affinage (dégelé)", f"ft_unfrozen_f10_s{HEAD_SEED}.md"), ("Tête temporelle", f"temporal_tl_f10_s{HEAD_SEED}.md")]:
    h, r = parse_curve(f)
    if r:
        ax[1].plot([x[0] for x in r], [x[1] for x in r], label=lab)
        ax[2].plot([x[0] for x in r], [x[2] for x in r], label=lab)
ax[1].set_yscale("log"); ax[1].set_title("Perte d'entraînement (échelle log)"); ax[1].set_xlabel("Époque"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].set_title("F1 macro de validation"); ax[2].set_xlabel("Époque"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
show_img("courbes_cv_s0.png")   # perte et exactitude de validation et de test (test : affichage seulement)

### Section 7 — Exigences (HS1 à HS4, EF2, OS4) : mesures

In [ ]:
for n in [f"temporal_tl_f10_s{HEAD_SEED}.md", "attack_seeds_tl_f10.md", "vote_tl_f10.md", "hs1_classes_tl_f10.md", "stability_tl_f10.md",
          "label_coverage_tl_f10.md", f"ef2_hosts_s{HEAD_SEED}.md", "ef2_novelty_tl_f10.md", f"replay_s{HEAD_SEED}.md", "latency.md"]:
    show_md(n, 7000)

### Section 8 — Explicabilité (TreeSHAP, GNNExplainer)

In [ ]:
for n in [f"surrogate_s{HEAD_SEED}.md", f"surrogate_tune_s{HEAD_SEED}.md", f"xai_shap_s{HEAD_SEED}.md", f"xai_shap_s{HEAD_SEED}cls_tuned.md",
          f"xai_gnn_tune_s{HEAD_SEED}.md", f"xai_gnn_s{HEAD_SEED}_gnn.md"]:
    show_md(n, 6000)
show_img(f"shap_summary_s{HEAD_SEED}.png")
show_img(f"gnn_explication_s{HEAD_SEED}.png")

### Section 9 — Robustesse et dérive

In [ ]:
show_md(f"adversarial_s{HEAD_SEED}.md")
show_md(f"drift_s{HEAD_SEED}.md")

### Diagnostics HS1

In [ ]:
for n in [f"volume_ddos_s{DIAG_SEED}.md", f"context_ddos_s{DIAG_SEED}.md", f"order_s{DIAG_SEED}_{DEVICE}.md"]:
    show_md(n, 5000)